<a href="https://colab.research.google.com/github/sejal-godbole/Federated-Learning/blob/main/FL_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To perform data pre-processing and partitioning for a Federated Learning system using a real dataset such as a student performance dataset (e.g., study hours, attendance, internal marks, final score). The dataset is cleaned by handling missing values and normalization, then split into multiple local datasets and distributed among participating devices or nodes so that each client trains on its own private data without sharing raw information.

Upload the Dataset

In [1]:
from google.colab import files

uploaded = files.upload()


Saving student_performance_100_rows.csv to student_performance_100_rows (1).csv


Import Required Libraries

In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split


Load the Dataset

In [3]:
# Load dataset
data = pd.read_csv("student_performance_100_rows.csv")

# Display first 5 rows
data.head()


,Study_Hours,Attendance,Internal_Marks,Final_Score
0,7,94,58,89
1,4,96,59,95
2,8,73,71,98
3,5,62,46,52
4,7,60,91,71


Check Dataset Information

In [4]:
# Dataset info
data.info()

# Check missing values
data.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Study_Hours     100 non-null    int64
 1   Attendance      100 non-null    int64
 2   Internal_Marks  100 non-null    int64
 3   Final_Score     100 non-null    int64
dtypes: int64(4)
memory usage: 3.3 KB


,0
Study_Hours,0
Attendance,0
Internal_Marks,0
Final_Score,0


Handle Missing Values - Data Cleaning

In [5]:
# Fill missing values with mean
data = data.fillna(data.mean())

# Verify again
data.isnull().sum()


,0
Study_Hours,0
Attendance,0
Internal_Marks,0
Final_Score,0


Separate Features and Target

In [6]:
# Features (Input)
X = data.drop("Final_Score", axis=1)

# Target (Output)
y = data["Final_Score"]

print("Features:")
print(X.head())

print("\nTarget:")
print(y.head())


Features:
   Study_Hours  Attendance  Internal_Marks
0            7          94              58
1            4          96              59
2            8          73              71
3            5          62              46
4            7          60              91

Target:
0    89
1    95
2    98
3    52
4    71
Name: Final_Score, dtype: int64


Normalize the Data

In [7]:
# Initialize scaler
scaler = MinMaxScaler()

# Apply scaling
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Show normalized data
X_scaled.head()


,Study_Hours,Attendance,Internal_Marks
0,0.750,0.894737,0.305085
1,0.375,0.947368,0.322034
2,0.875,0.342105,0.525424
3,0.500,0.052632,0.101695
4,0.750,0.000000,0.864407


Split into Train and Test

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

print("Training size:", len(X_train))
print("Testing size:", len(X_test))


Training size: 80
Testing size: 20


Create Federated Clients - Partioning

In [9]:
def create_federated_clients(X, y, num_clients=5):

    client_data = []

    data_size = len(X)
    samples_per_client = data_size // num_clients

    for i in range(num_clients):

        start = i * samples_per_client
        end = (i + 1) * samples_per_client

        X_client = X.iloc[start:end]
        y_client = y.iloc[start:end]

        client_data.append((X_client, y_client))

    return client_data


Split Data for Clients

In [10]:
NUM_CLIENTS = 5

clients = create_federated_clients(X_train, y_train, NUM_CLIENTS)

print("Number of Clients:", len(clients))


Number of Clients: 5


Verify Each Client's Data

In [11]:
for i, (Xc, yc) in enumerate(clients):

    print(f"\nClient {i+1}")
    print("Data Size:", len(Xc))

    print("Features Sample:")
    print(Xc.head(2))

    print("Target Sample:")
    print(yc.head(2))



Client 1
Data Size: 16
Features Sample:
    Study_Hours  Attendance  Internal_Marks
55        0.625    0.815789        0.813559
88        0.875    0.394737        0.542373
Target Sample:
55    91
88    58
Name: Final_Score, dtype: int64

Client 2
Data Size: 16
Features Sample:
    Study_Hours  Attendance  Internal_Marks
66        0.875    0.710526        0.474576
65        0.500    0.026316        0.406780
Target Sample:
66    53
65    77
Name: Final_Score, dtype: int64

Client 3
Data Size: 16
Features Sample:
    Study_Hours  Attendance  Internal_Marks
17        0.625    0.815789        0.372881
38        0.125    0.710526        0.983051
Target Sample:
17    79
38    96
Name: Final_Score, dtype: int64

Client 4
Data Size: 16
Features Sample:
    Study_Hours  Attendance  Internal_Marks
61        1.000    0.052632        0.576271
97        0.625    0.473684        0.491525
Target Sample:
61    46
97    67
Name: Final_Score, dtype: int64

Client 5
Data Size: 16
Features Sample:
    Stu

Save Client Data

In [12]:
for i, (Xc, yc) in enumerate(clients):

    client_df = Xc.copy()
    client_df["Final_Score"] = yc.values

    filename = f"client_{i+1}_data.csv"

    client_df.to_csv(filename, index=False)

    print(filename, "saved")


client_1_data.csv saved
client_2_data.csv saved
client_3_data.csv saved
client_4_data.csv saved
client_5_data.csv saved
